# Phase 111: replication and decoded recovery on IBM hardware

Runs the frozen Phase 111 design (`DESIGN.md`) on ibm_fez. Nothing here compiles, chooses or checks circuits by
itself: all run logic is in `p111_runner.py`, which is frozen and hashed with everything else; this notebook checks
that file's hash and then only calls it.

**Jobs, in order, each submitted once:**
1. **Part A2**, replication of Test B: the archived compiled circuits (`testB_transpiled.qpy`), 20 × 4,000 shots.
2. **Part B pilot**: the six circuits of D1 and D2 plus their calibrations, 1,000 shots each. No abort on the values;
   only a technical failure stops the run.
3. **Part B**: all decoding, baseline and calibration circuits, at the shot count fixed by the power check.

**Before running:**
- Put the frozen files (listed in `FREEZE_MANIFEST.txt`) in `MyDrive/phase111`.
- Verify the OpenTimestamps proof of `FREEZE_MANIFEST.txt` (for example at opentimestamps.org) and put the result in
  `TIMESTAMP_CHECK.json`: `{"manifest_sha256": ..., "attested_utc": ..., "bitcoin_block": ..., "verified_with": ...,
  "verified_utc": ...}`.
- Have the manifest's SHA-256 from the timestamped commit at hand, and set `INSTANCE` in the connection cell.

**Safeguards** (in `p111_runner.py`): every frozen file must match the timestamped manifest; the experiment ID derived
from it is in every job tag and job record; just before each submission all hashes, today's target, the shot count and
the order of steps are checked again and you confirm the QPU budget; retrieval checks the job's tag, backend, shots and
submitted circuits. After a disconnect, rerun the cells up to the connection, then the cell of the step you were on;
nothing is resubmitted. After the run, save a copy of this executed notebook with the results.

In [1]:
# The versions that built the frozen circuits: QPY files need a Qiskit at least as new as the writer.
!pip -q install "qiskit==2.6.1" "qiskit-ibm-runtime==0.51.0"

In [2]:
import os, sys, hashlib, getpass
OUTDIR = '/content/drive/MyDrive/phase111'
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:                       # not in Colab: use a local folder of the same name
    print('Drive not mounted:', e)
    OUTDIR = 'phase111'
print('working folder:', OUTDIR)

Mounted at /content/drive
working folder: /content/drive/MyDrive/phase111


## Check the frozen files
Checks `p111_runner.py` and the modules it imports against the manifest before importing them; the runner then checks
everything else.

In [3]:
STAMPED = input('SHA-256 of FREEZE_MANIFEST.txt in the timestamped commit: ').strip().lower()
_sha = lambda n: hashlib.sha256(open(os.path.join(OUTDIR, n), 'rb').read()).hexdigest()
assert _sha('FREEZE_MANIFEST.txt') == STAMPED, 'this FREEZE_MANIFEST.txt is not the timestamped one'
_listed = {}
for _line in open(os.path.join(OUTDIR, 'FREEZE_MANIFEST.txt')):
    if _line.strip() and not _line.startswith('#'):
        _h, _n = _line.split(None, 1)
        _listed[_n.strip()] = _h
for _f in ('p111_runner.py', 'p111_verify.py', 'p111_analyse.py'):
    assert _listed.get(_f) == _sha(_f), f'{_f} does not match the manifest'
sys.path.insert(0, OUTDIR)
import p111_runner as RUN
EXP = RUN.Experiment(OUTDIR, STAMPED)

SHA-256 of FREEZE_MANIFEST.txt in the timestamped commit: f1b99e67ee15ed148b2550ea78d37c5094ff1fbdea9a4e8da1a8ec45532260d6
all 16 frozen files match the timestamped manifest (attested 2026-10-25T18:00:31+00:00) | experiment p111-f1b99e67ee15
A2: 20 circuits x 4000 shots | pilot: 14 circuits x 1000 shots | B: 60 circuits x 4000 shots


## Connect to IBM Quantum
Uses the `IBM_QUANTUM_TOKEN` environment variable if set, otherwise asks for your API key (not stored). Set
`INSTANCE` to your own instance CRN.

In [4]:
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
CHANNEL = "ibm_cloud"
INSTANCE = "<your IBM Quantum instance CRN>"
token = os.environ.get('IBM_QUANTUM_TOKEN') or getpass.getpass("IBM Quantum API key: ")
EXP.connect(QiskitRuntimeService(channel=CHANNEL, token=token, instance=INSTANCE), "ibm_fez", SamplerV2)

IBM Quantum API key: ··········


qiskit_runtime_service._discover_account:WARNING:2026-10-25 19:11:09,680: Loading account with the given token. A saved account will not be used.


using ibm_fez | pending jobs: 0
existing A2 jobs for this experiment: []
existing pilot jobs for this experiment: []
existing B jobs for this experiment: []


## Target check (no QPU time)
Reports whether today's backend supports every frozen circuit (checked again before each submission). If the archived
Test B circuits fail, Part A2 needs the A2b procedure; if the Part B circuits fail, they are recompiled offline by the
frozen procedure, re-verified and re-stamped. This notebook does not recompile.

In [5]:
EXP.target_report()

A2: supported by today's target
pilot: supported by today's target
B: supported by today's target


## Part A2: replication of Test B (archived circuits)

In [6]:
EXP.submit_once('A2')

IBM usage report: {'instance_id': '<your IBM Quantum instance CRN>', 'plan_id': '<redacted>', 'usage_consumed_seconds': 27, 'usage_consumed_ru': 0.429439, 'usage_period': {'start_time': '2026-09-27T19:11:31.792Z', 'end_time': '2026-10-25T19:11:31.792Z'}, 'usage_limit_seconds': 600, 'usage_limit_ru': 10, 'usage_limit_reached': False, 'usage_remaining_seconds': 573}
A2: 20 circuits x 4000 shots, up to 120 QPU seconds. Type YES to submit: YES
A2 JOB ID (write it down): dbf58r8m94ac738ssbi0
saved A2_jobid.json


'dbf58r8m94ac738ssbi0'

In [7]:
A2_OUT = EXP.analyse('A2')        # retrieve and analyse (safe to rerun)

A2: submitted circuits match the frozen fingerprints
saved A2_raw_counts.json
A2: retrieved, usage 23
  main_B_not_A/A   E 0.828 (se 0.018)
  main_B_not_A/B   E 1.383 (se 0.016)
  both/A           E 1.558 (se 0.014)
  both/B           E 1.611 (se 0.013)
  neither/A        E 0.863 (se 0.018)
  neither/B        E 0.853 (se 0.018)
  B1: +0.383, z 23.8, holds
  B2: +0.558, z 40.4, holds
  B3: +0.555, z 22.8, holds
PART A2: REPLICATED
saved A2_result.json


## Part B pilot (fixed; no abort on the values)

In [8]:
EXP.submit_once('pilot')

IBM usage report: {'instance_id': '<your IBM Quantum instance CRN>', 'plan_id': '<redacted>', 'usage_consumed_seconds': 50, 'usage_consumed_ru': 0.804424, 'usage_period': {'start_time': '2026-09-27T19:12:40.981Z', 'end_time': '2026-10-25T19:12:40.981Z'}, 'usage_limit_seconds': 600, 'usage_limit_ru': 10, 'usage_limit_reached': False, 'usage_remaining_seconds': 550}
pilot: 14 circuits x 1000 shots, up to 120 QPU seconds. Type YES to submit: YES
PILOT JOB ID (write it down): dbf59bnsstms73fd4ntg
saved pilot_jobid.json


'dbf59bnsstms73fd4ntg'

In [9]:
P_OUT = EXP.analyse('pilot')      # only a technical failure (an error raised here) stops the run

pilot: submitted circuits match the frozen fingerprints
saved pilot_raw_counts.json
pilot: retrieved, usage 6
  main_B_not_A/B   F 0.760 (se 0.010, ideal 1.0)  corrected 0.788
  both/A           F 0.694 (se 0.011, ideal 1.0)  corrected 0.716
saved pilot_result.json


## Part B: decoded recovery

In [10]:
EXP.submit_once('B')

IBM usage report: {'instance_id': '<your IBM Quantum instance CRN>', 'plan_id': '<redacted>', 'usage_consumed_seconds': 56, 'usage_consumed_ru': 0.89379, 'usage_period': {'start_time': '2026-09-27T19:13:54.224Z', 'end_time': '2026-10-25T19:13:54.224Z'}, 'usage_limit_seconds': 600, 'usage_limit_ru': 10, 'usage_limit_reached': False, 'usage_remaining_seconds': 544}
B: 60 circuits x 4000 shots, up to 300 QPU seconds. Type YES to submit: YES
B JOB ID (write it down): dbf59tvpc3tc739mp16g
saved B_jobid.json


'dbf59tvpc3tc739mp16g'

In [11]:
B_OUT = EXP.analyse('B')          # retrieve and analyse (safe to rerun)

B: submitted circuits match the frozen fingerprints
saved B_raw_counts.json
B: retrieved, usage 65
  main_B_not_A/B   F 0.761 (se 0.005, ideal 1.0)  corrected 0.788
  main_B_not_A/A   F 0.431 (se 0.006, ideal 0.5)  corrected 0.439
  both/A           F 0.683 (se 0.006, ideal 1.0)  corrected 0.705
  both/B           F 0.808 (se 0.005, ideal 1.0)  corrected 0.819
  neither/A        F 0.470 (se 0.006, ideal 0.5)  corrected 0.477
  neither/B        F 0.431 (se 0.006, ideal 0.5)  corrected 0.467
  baseline 21-16: F 0.892 (se 0.004)
  baseline 20-16: F 0.866 (se 0.004)
  baseline 0-4: F 0.784 (se 0.005)
  baseline 23-4: F 0.920 (se 0.003)
  baseline 36-40: F 0.884 (se 0.004)
  baseline 22-43: F 0.816 (se 0.004)
  D1: +0.261, z 52.1, holds
  D2: +0.183, z 32.8, holds
  D3: +0.330, z 41.4, holds
PART B: PASS
saved B_result.json


## Job record
Lists every job on this instance since a day before now (any status), for `RUNLOG.md`.

In [12]:
EXP.job_record()

{'job_id': 'dbf59tvpc3tc739mp16g', 'status': 'DONE', 'tags': ['p111-f1b99e67ee15-B'], 'created': '2026-10-25 19:13:59.862197+00:00'}
{'job_id': 'dbf59bnsstms73fd4ntg', 'status': 'DONE', 'tags': ['p111-f1b99e67ee15-pilot'], 'created': '2026-10-25 19:12:46.898871+00:00'}
{'job_id': 'dbf58r8m94ac738ssbi0', 'status': 'DONE', 'tags': ['p111-f1b99e67ee15-A2'], 'created': '2026-10-25 19:11:41.736806+00:00'}
saved job_record.json
